# CICIDS2017 Hardware Experiments (IBM Sherbrooke)

This notebook runs quantum IDS experiments on **IBM Sherbrooke fake backend** using the CICIDS2017 dataset.

**Key Features:**
- Simulates IBM Sherbrooke quantum hardware (127 qubits)
- Realistic noise model from actual IBM hardware
- Shot-based execution (1024 shots)
- Samples 20,000 records from CICIDS2017 with memory-efficient loading
- Uses 10 features/qubits (PCA reduced)

**Models:**
- Classical: LogReg, RandomForest, XGBoost/LightGBM
- Quantum (Hardware): QSVM, QNN variants, QKNN
- Fusion: Meta-ensemble models

**Configuration Options:**
1. `config_fast`: QSVM + QNN only (~2-3 hours)
2. `config_full`: All models including QKNN (~4-6 hours)

In [ ]:
import os
import sys
import pandas as pd

# Add hardware pipeline to path
sys.path.append('/home/ritvik/Python/Copilot Files/Hardware/utils')
from utils.hardware_quantum_pipeline import HardwareConfig, run_dataset_experiment_hardware

## Configuration

CICIDS2017-specific configuration:
- **10 features** (qubits) instead of 13
- **20,000 samples** from full dataset
- Same shot count (1024) and backend (IBM Torino)

In [ ]:
# Option 1: QSVM + QNN only (faster execution)
config_fast = HardwareConfig(
    seed=42,
    num_features=10,  # 10 qubits for CICIDS2017
    cicids_sample_n=20000,  # Sample 20k rows
    shots=1024,
    backend_name="fake_sherbrooke",
    optimization_level=2,
    max_workers=max(1, (os.cpu_count() or 4) // 2),
    paper_fontsize=12,
    export_dpi=600,
    show_progress=True,
    # QSVM configuration (10 batches)
    qsvm_train_size=300,
    qsvm_test_size=100,
    # QNN configuration
    qnn_train_size=3000,
    qnn_test_size=1000,
    # Model selection
    run_qnn_models=True,
    run_qknn=False,  # Skip QKNN for faster execution
)

# Option 2: All models (complete comparison)
config_full = HardwareConfig(
    seed=42,
    num_features=10,
    cicids_sample_n=20000,
    shots=1024,
    backend_name="fake_sherbrooke",
    optimization_level=2,
    max_workers=max(1, (os.cpu_count() or 4) // 2),
    paper_fontsize=12,
    export_dpi=600,
    show_progress=True,
    qsvm_train_size=300,
    qsvm_test_size=100,
    qnn_train_size=3000,
    qnn_test_size=1000,
    run_qnn_models=True,
    run_qknn=True,  # Include QKNN
)

# Select which config to use
config = config_fast  # Change to config_full for complete comparison
config

## Run Hardware Experiment

This cell runs the complete experiment on IBM Sherbrooke fake backend.

**Expected runtime:**
- config_fast: ~2-3 hours
- config_full: ~4-6 hours

**Data loading:**
- CICIDS2017 dataset is large (~2.5M rows)
- Uses memory-efficient chunked loading
- Samples 20,000 rows stratified by attack type

**What happens:**
1. Load and preprocess CICIDS2017 (20k sampled rows)
2. Run classical baselines (LogReg, RF, XGBoost)
3. Run QSVM on hardware (10 batches × 300/100 samples)
4. Run QNN models on hardware (if enabled)
5. Run QKNN on hardware (if enabled)
6. Generate fusion meta-models
7. Export ROC/PR plots
8. Save results to Hardware/results/

In [ ]:
results_df = run_dataset_experiment_hardware(
    dataset_name='cicids2017_hardware',
    config=config,
    cicids_folder_path='/home/ritvik/Python/Datasets/CICIDS2017'
)
results_df

## Results Summary

Display key metrics sorted by AUPRC.

**CICIDS2017 characteristics:**
- More diverse attack types than train_test_network
- Higher class imbalance
- May show different noise sensitivity patterns

In [ ]:
summary_cols = [
    'Dataset', 'Model', 'Accuracy', 'F1', 'AUPRC', 'ROC_AUC',
    'TPR@0.1%FPR', 'TPR@1%FPR', 'Backend', 'Shots', 'Status'
]
available_cols = [c for c in summary_cols if c in results_df.columns]
results_df[available_cols].sort_values(by=['AUPRC'], ascending=False, na_position='last')

## Hardware vs. Simulator Comparison

Compare hardware results with simulator results for CICIDS2017.

In [ ]:
# Load simulator results for comparison
sim_results_path = '/home/ritvik/Python/Copilot Files/results/cicids2017_metrics_summary.csv'

if os.path.exists(sim_results_path):
    sim_df = pd.read_csv(sim_results_path)
    
    # Compare quantum models only
    quantum_models = ['QSVM', 'QNN_Base', 'QNN_Reupload', 'VQC_AltAnsatz', 'QKNN']
    
    comparison = []
    for model in quantum_models:
        sim_row = sim_df[sim_df['Model'] == model]
        hw_row = results_df[results_df['Model'] == model]
        
        if not sim_row.empty and not hw_row.empty:
            comparison.append({
                'Model': model,
                'Sim_Accuracy': sim_row['Accuracy'].values[0],
                'HW_Accuracy': hw_row['Accuracy'].values[0],
                'Accuracy_Delta': hw_row['Accuracy'].values[0] - sim_row['Accuracy'].values[0],
                'Sim_F1': sim_row['F1'].values[0],
                'HW_F1': hw_row['F1'].values[0],
                'F1_Delta': hw_row['F1'].values[0] - sim_row['F1'].values[0],
                'Sim_AUPRC': sim_row['AUPRC'].values[0],
                'HW_AUPRC': hw_row['AUPRC'].values[0],
                'AUPRC_Delta': hw_row['AUPRC'].values[0] - sim_row['AUPRC'].values[0],
            })
    
    comparison_df = pd.DataFrame(comparison)
    print("\n=== Simulator vs. Hardware Comparison (CICIDS2017) ===")
    print("Negative delta = hardware degradation due to noise\n")
    comparison_df
else:
    print("Simulator results not found. Run 02_cicids2017_experiments.ipynb first.")

## Cross-Dataset Comparison

Compare hardware performance across both datasets.

In [ ]:
# Load train_test_network hardware results
ttn_hw_path = '/home/ritvik/Python/Copilot Files/Hardware/results/train_test_network_hardware_metrics_summary.csv'

if os.path.exists(ttn_hw_path):
    ttn_hw_df = pd.read_csv(ttn_hw_path)
    
    # Combine both datasets
    combined = pd.concat([
        ttn_hw_df[['Model', 'Accuracy', 'F1', 'AUPRC']].assign(Dataset='Train_Test_Network'),
        results_df[['Model', 'Accuracy', 'F1', 'AUPRC']].assign(Dataset='CICIDS2017')
    ])
    
    # Pivot for side-by-side comparison
    pivot = combined.pivot_table(
        index='Model',
        columns='Dataset',
        values=['Accuracy', 'F1', 'AUPRC']
    )
    
    print("\n=== Hardware Performance Across Datasets ===")
    pivot
else:
    print("Train/Test Network hardware results not found. Run 01_hardware_train_test_network.ipynb first.")

## Next Steps

1. **Analyze hardware characteristics**: Run `03_hardware_analysis.ipynb` for detailed IBM Sherbrooke analysis
2. **Review plots**: Check `Plots PDF/` and `Images/` folders for ROC/PR curves
3. **Conference paper**: Use results from hardware analysis notebook for publication
4. **Compare datasets**: Analyze how different datasets respond to hardware noise